# Advanced Chemical Reactor Engineering - Group: Stirred not Shaken
### CSTR Reactor Simulation: Glucose → Fructose → HMF → FDCA

This notebook simulates a Continuous Stirred Tank Reactor (CSTR) for the catalytic oxidation of glucose to FDCA (furandicarboxylic acid).

### Reaction pathway
$$\text{Glucose} \xrightarrow{k_1} \text{Fructose} \xrightarrow{k_3} \text{HMF} \xrightarrow{k_6} \text{FDCA}$$

Side reactions from HMF:
- $\text{HMF} \xrightarrow{k_4} \text{Humins}$
- $\text{HMF} \xrightarrow{k_5} \text{Levulinic acid + Formic acid}$

### Three-phase system
The reactor operates as a gas–liquid–solid (G-L-S) system:
- Gas phase: $O_2$ bubbles
- Liquid phase: aqueous (sugars/HMF) + organic MIBK solvent (HMF extraction)
- Solid phase: catalyst particles

Mass transfer steps: Gas $\rightarrow$ Organic (GL), Aqueous $\rightarrow$ Organic (LL), Organic $\rightarrow$ Catalyst (LS)

### 0. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import warnings
warnings.filterwarnings('ignore')

### 1. Physical Properties

All properties are evaluated at the operating temperature T = 140 °C (413.15 K).

#### Diffusivities — Wilke–Chang correlation
The diffusivity is corrected from a reference value at 298 K to the operating conditions using:
$$D(T, \mu) = D_{\text{ref}} \cdot \frac{T}{T_{\text{ref}}} \cdot \frac{\mu_{\text{ref}}}{\mu}$$

This accounts for both the increase in thermal energy (higher T → faster diffusion) and the decrease in viscosity at elevated temperature (lower μ -> less resistance to molecular motion).

In [ ]:
T, g, R = 160 + 273.15, 9.81, 8.3145   # K, m/s^2, J/mol/K
rho     = 780.0     # kg_org/m_org^3  density MIBK
mu      = 0.35e-3   # Pa.s    dynamic viscosity MIBK
nu      = mu / rho  # m_org^2/s   kinematic viscosity
sigma   = 12e-3     # N/m_org     interfacial tension MIBK/water

# Diffusivities corrected to operating T and mu via Wilke-Chang
mu_ref = 0.585e-3
D_HMF  = 6.0e-10 * (T/298) * (mu_ref/mu)   # m_org^2/s  HMF in MIBK
D_O2   = 3.5e-9  * (T/298) * (mu_ref/mu)   # m_org^2/s  O2 in MIBK
D_FDCA = 3.5e-9  * (T/298) * (mu_ref/mu)   # m_org^2/s  FDCA in MIBK

print(f"T = {T:.2f} K")
print(f"D_HMF  = {D_HMF:.3e} m_org^2/s")
print(f"D_O2   = {D_O2:.3e} m_org^2/s")
print(f"D_FDCA = {D_FDCA:.3e} m_org^2/s")

### 2. Reactor Geometry and Phase Volume Fractions

The 100 $m^3$ reactor is divided into four phases. Volume fractions are defined as $m^3_{\text{phase}} / m^3_{\text{reactor}}$:

| Phase | Symbol | Fraction | Volume (m³) | Units (Fractional) |
|-------|--------|----------|-------------|--------------------|
| Gas (O₂ bubbles) | $\varepsilon_{\text{g}}$ | 0.20 | 20 | $m_{\text{g}}^3 / m_{\text{R}}^3$ |
| Catalyst particles | $\varepsilon_{\text{s}}$ | 0.10 | 10 | $m_{\text{s}}^3 / m_{\text{R}}^3$ |
| Ral liquid (aq + org) | $\varepsilon_{\text{l}}$ | 0.70 | 70 | $m_{\text{L}}^3 / m_{\text{R}}^3$ |
| &nbsp;&nbsp;Aqueous (1:2 split) | $\varepsilon_{\text{aq}}$ | 0.233 | 23.33 | $m_{\text{aq}}^3 / m_{\text{L}}^3$ |
| &nbsp;&nbsp;Organic MIBK (1:2 split) | $\varepsilon_{\text{org}}$ | 0.467 | 46.67 | $m_{\text{org}}^3 / m_{\text{L}}^3$ |

The **aqueous:organic ratio of 1:2** reflects the extraction design — HMF partitions preferentially into MIBK, which drives the dehydration equilibrium forward.

The agitator diameter follows the standard rule $D_A / D_T = 1/3$ for Rushton turbines.

In [ ]:
V_total = 100.0                     # m_R^3 reactive volume
eps_g = 0.20                        # m_g^3/m_R^3 bubble volume fraction
eps_s = 0.10                        # m_s^3/m_R^3 particles volume fraction
eps_l_set = 1.0 - eps_g - eps_s     # m_l^3/m_R^3 total liquids volume fraction
eps_aq = 1/3                        # m_aq^3/m_l^3 aquatic phase faction
eps_org = 1 - eps_aq                # m_org^3/m_l^3 organic phase fraction

V_g   = eps_g * V_total         # m_g^3   bubble gas phase
V_p   = eps_s * V_total         # m_s^3   catalyst particles
V_liq = eps_l_set * V_total     # m_l^3   total liquid (aq + org)
V_aq  = V_liq * eps_aq          # m_aq^3   aqueous phase
V_org = V_liq * eps_org         # m_org^3   organic phase

D_T  = (4 * V_total / np.pi)**(1/3)    # m    tank diameter
D_A  = D_T / 3                         # m    agitator (D_A/D_T = 1/3)
A_cs = np.pi / 4 * D_T**2              # m^2  cross-section

C_Glu_feed = 1500.0   # mol/m_aq^3  glucose feed concentration
m_AO = 0.77           # [-]  HMF partition coefficient aq -> org <====== check the units of this partition coefficient!!!

tau   = 3600          # s residence time (1 h)
F_aq  = V_aq  / tau   # m_aq^3/s  aqueous flow rate
F_org = V_org / tau   # m_org^3/s organic flow rate
F_g   = V_g / tau     # m_g^3/s   gas flow rate  

print(f"Residence time tau = {tau/3600:.1f} h")
print(f"F_aq  = {F_aq*1000:.2f} L/s")
print(f"F_org = {F_org*1000:.2f} L/s")
print(f"F_org = {F_g*1000:.2f} L/s\n")

print(f"Tank diameter:   D_T = {D_T:.2f} m")
print(f"Agitator diam:   D_A = {D_A:.2f} m")
print(f"Phase volumes:   V_g={V_g:.1f}  V_aq={V_aq:.2f}  V_org={V_org:.2f}  V_p={V_p:.1f} m^3")
print(f"Sum check:       {V_g+V_aq+V_org+V_p:.2f} m^3  (should be {V_total})")

### 3. Agitation, Gas Flow, and Power Input

#### Minimum agitation speed
The impeller must rotate fast enough to disperse gas bubbles. The minimum speed $N_{RM}$ (Calderbank criterion) scales with tank/agitator geometry:
$$N_{RM} = 0.22 \cdot \frac{D_T^{1.5}}{D_A^2}$$

#### Power dissipation
Total power input has two contributions:

1. Impeller power (6-blade Rushton turbine, power number $N_p = 5$):
$$P_L = N_p \cdot \rho \cdot N_R^3 \cdot D_A^5$$

2. Gas expansion power (isothermal expansion of rising bubbles):
$$P_G = v_{SG} \cdot \rho \cdot g \cdot V_{\text{total}}$$

The specific energy dissipation rate $\varepsilon = P/V\rho$ ($m^2$/$s^3$) governs turbulent mass transfer and is used in the mass transfer correlations below.

In [ ]:
N_p =  5.0              # -      wanted power number (6-blade Rushton turbine)

v_SG = 4 * F_g / (np.pi*D_T**2)     # m/s     superficial gas velocity
P_G  = v_SG * rho * g * V_total     # W       gas power
PV = (eps_g / 0.27 / v_SG**0.67) ** (1/0.31) # power per unit of volume
P_L  = PV*V_liq -  P_G              # W           impeller power

N_RM = 0.22 * D_T**1.5 / D_A**2           # rev/s  minimal needed speed (Calderbank)
N_R  = (P_L / rho / N_p / D_A**5)**(1/3)  # rev/s  minimal wanted stirrer angular speed
print(f"N_R = {N_R:.2f} rev/s  >  N_RM = {N_RM:.2f} rev/s: {N_R > N_RM}")

eps  = PV / rho                       # m_l^2/s^3     specific energy dissipation

# Gas holdup and bubble diameter (Calderbank correlations)
d_b   = 4.15*(sigma/rho)**0.6 * (P_L/V_total)**(-0.4) * v_SG**0.5 + 9e-4  # m_g^3/m_int^2 surface average bubble diameter (Sauter Diameter)

print(f"P/V  = {PV:.1f} W/m^3")
print(f"eps  = {eps:.4f} m^2/s^3")
print(f"eps_g (gas holdup) = {eps_g:.3f}")
print(f"d_b  (bubble diam) = {d_b*1e3:.2e} m_g^3/m_int^2")

### 4. Mass Transfer Coefficients

Three interfacial mass transfer steps are modelled, each with a volumetric coefficient $k_La$ ($s^{-1}$).

#### A1 — Gas → Organic liquid (O₂ absorption)
Yagi & Yoshida (1975) correlation for the liquid-film coefficient:
$$k_La_{GL} = 0.06 \cdot Re^{1.5} \cdot Fr^{0.19} \cdot Sc^{0.5} \cdot Ca^{0.6} \cdot V_r^{0.32} \cdot \frac{D_{O_2}}{D_A^2}$$

Dimensionless groups:

$Re = N_R D_A^2 \rho/\mu$

$Fr = N_R^2 D_A/g$

$Sc = \mu/(\rho D)$

$Ca = \mu v_{SG}/\sigma$ 

$V_r = N_R D_A/v_{SG}$

The interfacial area per unit volume is:
$$a_{GL} = \frac{6\,\varepsilon_g}{d_b}$$

#### A2 — Aqueous → Organic liquid (HMF and FDCA)
Armenante & Kirwan (1989) — Sherwood number for droplets in turbulent flow:
$$Sh = 2 + 0.36 \cdot Re_{LL}^{0.75} \cdot Sc^{0.33}$$
where $Re_{LL} = (\varepsilon\, d_{\text{drop}}^4/\nu^3)^{0.25}$ is a turbulence-based Reynolds number.

Droplet diameter from the turbulent Kolmogorov length scale:
$$d_{\text{drop}} = \left(\frac{\sigma}{\rho\,\varepsilon^{2/3}}\right)^{0.6}$$

#### A3 — Organic liquid → Catalyst particle (HMF, $O_2$ and FDCA)
Same Armenante & Kirwan correlation applied to solid particles of diameter $d_p = 5\,\mu\text{m}_\text{s}$:
$$a_{LS} = \frac{6\,\varepsilon_s}{d_p}$$

In [ ]:
# Helper: Armenante & Kirwan (1989) Sherwood number
def Sh_AK(eps, d, nu, mu, rho, D):
    Re = (eps * d**4 / nu**3)**0.25
    Sh = 2 + 0.36 * Re**0.75 * (mu / (rho * D))**0.33
    return Sh

# A1: Gas-liquid (O2: gas -> organic) — Yagi & Yoshida (1975)
kLa_GL = (0.06
          * (N_R * D_A**2 * rho / mu)**1.5       # Re
          * (N_R**2 * D_A / g)**0.19              # Fr
          * (mu / (rho * D_O2))**0.5              # Sc
          * (mu * v_SG / sigma)**0.6              # Ca
          * (N_R * D_A / v_SG)**0.32              # Vr
          * D_O2 / D_A**2)
a_GL = 6 * eps_g / d_b   # m_int^2/m_R^3

# A2: Liquid-liquid (HMF & FDCA: aq -> organic) — Armenante & Kirwan (1989)
d_drop = np.clip((sigma / (rho * eps**(2/3)))**0.6, 2e-4, 3e-3)  # m_aq
a_LL   = 6 * eps_aq / d_drop                                     # m_aq^2/m_org^3

kLa_LL_HMF  = Sh_AK(eps, d_drop, nu, mu, rho, D_HMF)  * D_HMF  / d_drop * a_LL  # s^-1
kLa_LL_FDCA = Sh_AK(eps, d_drop, nu, mu, rho, D_FDCA) * D_FDCA / d_drop * a_LL  # s^-1

# A3: Liquid-solid (HMF & O2: organic -> catalyst) — Armenante & Kirwan (1989)
d_p  = 5e-6                     # m_p  catalyst diameter
a_LS = 6 * eps_s / d_p          # m_int^2/m_cat^3

kLa_LS_HMF = Sh_AK(eps, d_p, nu, mu, rho, D_HMF) * D_HMF / d_p * a_LS  # s^-1
kLa_LS_O2  = Sh_AK(eps, d_p, nu, mu, rho, D_O2)  * D_O2  / d_p * a_LS  # s^-1
kLa_LS_FDCA  = Sh_AK(eps, d_p, nu, mu, rho, D_FDCA)  * D_FDCA  / d_p * a_LS  # s^-1

print(f"kLa_GL      (O2   gas->org) = {kLa_GL:.4f} s^-1")
print(f"kLa_LL_HMF  (HMF  aq->org)  = {kLa_LL_HMF:.4f} s^-1")
print(f"kLa_LL_FDCA (FDCA aq->org)  = {kLa_LL_FDCA:.4f} s^-1")
print(f"kLa_LS_HMF  (HMF  org->cat) = {kLa_LS_HMF:.2f}  s^-1")
print(f"kLa_LS_O2   (O2   org->cat) = {kLa_LS_O2:.2f}  s^-1")
print(f"kLa_LS_FDCA (FDCA org->cat) = {kLa_LS_FDCA:.2f}  s^-1")

### 5. Reaction Kinetics

#### Isomerisation and dehydration (aqueous phase)
All rate constants are assumned to be first-order ($s^{-1}$):

| Reaction | Symbol | Value (s⁻¹) |
|----------|--------|-------------|
| Glucose $\rightarrow$ Fructose | $k_1$ | 0.104/60 |
| Fructose $\rightarrow$ Glucose (reverse) | $k_2$ | 0.052/60 |
| Fructose $\rightarrow$ HMF | $k_3$ | 0.286/60 |
| HMF $\rightarrow$ Humins | $k_4$ | 0.013/60 |
| HMF $\rightarrow$ LA + FA | $k_5$ | 0.031/60 |

#### Oxidation to FDCA (catalyst particle surface)
HMF oxidation proceeds via two parallel series paths:
- Path A: HMF $\rightarrow$ DFF $\rightarrow$ FFCA $\rightarrow$ FDCA
- Path B: HMF $\rightarrow$ HFCA $\rightarrow$ FFCA $\rightarrow$ FDCA

For each path, the series steps are combined using the resistance-in-series formula $1/k_{\text{eff}} = 1/k_1 + 1/k_2$, then the two parallel paths are added:
$$k_6 = \frac{1}{\frac{1}{k_{\text{HMF→DFF}}} + \frac{1}{k_{\text{DFF→FFCA}}}} + \frac{1}{\frac{1}{k_{\text{HMF→HFCA}}} + \frac{1}{k_{\text{HFCA→FFCA}}}}$$

#### Thiele modulus and internal effectiveness factor
For a spherical catalyst particle with 1st-order reaction in HMF, the Thiele modulus is:
$$\phi = \frac{d_p}{6}\sqrt{\frac{k_6}{D_{\text{HMF}}}}$$

The internal effectiveness factor (ratio of actual to maximum reaction rate) is:
$$\eta = \frac{1}{\phi}\left(\frac{1}{\tanh(3\phi)} - \frac{1}{3\phi}\right)$$

When $\phi \ll 1$: $\eta \approx 1$ → no internal diffusion limitation.

In [ ]:
k1, k2 = 0.104/60, 0.052/60    # s^-1  Glu <-> Fru (isomerisation)
k3      = 0.286/60             # s^-1  Fru -> HMF  (dehydration)
k4      = 0.013/60             # s^-1  HMF -> Humins
k5      = 0.031/60             # s^-1  HMF -> LA + FA

# Oxidation: two parallel series paths (resistance-in-series + parallel combination)
# Path A (via DFF):   1/k_eff_A = 1/k_HMF_DFF + 1/k_DFF_FFCA
# Path B (via HFCA):  1/k_eff_B = 1/k_HMF_HFCA + 1/k_HFCA_FFCA
k6 = 1/(1/0.0556 + 1/0.01576) + 1/(1/0.0298 + 1/1.58e-3)   # s^-1 Overall reaction rate on catalyst
k6_ = k6 

# Thiele modulus & internal effectiveness factor (sphere, 1st order)
phi = (d_p / 6) * np.sqrt(k6 / D_HMF)
eta = (1/np.tanh(3*phi) - 1/(3*phi)) / phi

print(f"k6  (oxidation, combined) = {k6:.4f} s^-1")
print(f"Thiele modulus  phi = {phi:.5f}")
print(f"Effectiveness   eta = {eta:.6f}  (phi << 1 -> no internal diffusion limitation)")

### 6. O₂ Saturation Concentration — Henry's Law

The driving force for gas–liquid O₂ transfer is $C_{O_2}^* - C_{O_2,\text{org}}$, where $C_{O_2}^*$ is the saturation concentration set by Henry's law.

#### Vapour pressures (Antoine equation)
The total pressure is 10 bar; the partial pressure of O₂ is:
$$P_{O_2} = P_{\text{total}} - P_{\text{vap,water}} - P_{\text{vap,MIBK}}$$

#### Henry's constant temperature correction (van't Hoff)
Henry's constant increases with temperature (O₂ becomes less soluble at higher T):
$$H(T) = H_{298} \cdot \exp\!\left[\frac{\Delta H_{\text{sol}}}{R}\left(\frac{1}{298} - \frac{1}{T}\right)\right]$$

#### Saturation concentration
$$C_{O_2}^* = \frac{P_{O_2}}{H(T)} \quad \left[\frac{mol}{m_\text{org}^3}\right]$$

In [ ]:
# Vapour pressures via Antoine equation
P_O2      = 10e5                                            # Pa    chosen partial pressure O2
P_aq_vap  = 10**(3.55959 - 643.748/(T - 198.043))*1e5       # Pa    partial pressure aqueous phase            
P_org_vap = 10**(3.95298 - 1254.095/(T - 71.537))*1e5       # Pa    partial pressure organic phase      
P_O2_min  = P_aq_vap + P_org_vap                            # Pa    minimum partial pressure of O2 gas required for flux into liquid phases
P_tot_min = P_aq_vap + P_org_vap + P_O2_min                 # Pa    total pressure in reactor required for it to work
  
# Henry's constant at 298 K, corrected to operating T
H_O2_298  = 101.3e3 / (8.71e-4 * (780e3/58.08))             # Pa.m_org^3/mol    solubility at 25C
H_O2      = H_O2_298 * np.exp(15e3/R * (1/298 - 1/T))       # Pa.m_org^3/mol    solubility at 160C
C_O2_sat  = P_O2 / H_O2                                     # mol/m_org^3       saturation concentration
 
print(f"P_vap water   = {P_aq_vap:>10.2e} Pa")
print(f"P_vap MIBK    = {P_org_vap:>10.2e} Pa\n")
print(f"P_O2 Minimal  = {P_O2_min:>10.2e} Pa")
print(f"P_tot Minimal = {P_tot_min:>10.2e} Pa\n")
print(f"P_O2 Chosen   = {P_O2:>10.2e} Pa\n")
print(f"H_O2          = {H_O2:>10.2e} Pa.m_org^3/mol")
print(f"C*_O2         = {C_O2_sat:>10.4e} mol/m_org^3")

if P_O2 < P_O2_min:
    raise ValueError(f"Chosen O2 partial pressure lower than vapour pressure. System will have no O2 flux to reaction")

In [ ]:
C_Glu_feed = 1500.0   # mol/m_aq^3  glucose feed concentration
m_AO = 0.77           # m_aq^3/m_org^3  HMF partition coefficient aq -> org

print(f"Residence time tau = {tau/3600:.1f} h")
print(f"F_aq  = {F_aq*1000:.2f} L/s")
print(f"F_org = {F_org*1000:.2f} L/s")

### 7. ODE System — 9 State Variables

The reactor contains four phases: aqueous, organic (MIBK), catalyst particle, and gas.
Each species in each phase is treated as a separate component — $C_{\text{HMF,aq}} \neq C_{\text{HMF,org}}$.

The general steady-state mole balance for any phase is:

$$\frac{dC_i}{dt} = \underbrace{\frac{F}{V}(C_{i,\text{feed}} - C_i)}_{\text{convective flow}} + \underbrace{\sum r_j}_{\text{reaction}} \pm \underbrace{J_{\text{MT}}}_{\text{mass transfer}}$$

#### State variables
| # | Symbol | Phase | Species |
|---|--------|-------|---------|
| 0 | $C_{\text{Glu}}$ | aqueous | Glucose |
| 1 | $C_{\text{Fru}}$ | aqueous | Fructose |
| 2 | $C_{\text{HMF,aq}}$ | aqueous | HMF |
| 3 | $C_{\text{HMF,org}}$ | organic | HMF |
| 4 | $C_{\text{HMF,p}}$ | catalyst | HMF |
| 5 | $C_{O_2\text{,org}}$ | organic | O₂ |
| 6 | $C_{O_2\text{,p}}$ | catalyst | O₂ |
| 7 | $C_{\text{Hum}}$ | aqueous | Humins |
| 8 | $C_{\text{LA}}$ | aqueous | Levulinic acid (= FA) |

---

#### Mole balances

**Aqueous phase** — well-mixed CSTR ($\tau = V_{\text{aq}}/F_{\text{aq}}$):

$$\frac{dC_{\text{Glu}}}{dt} = \frac{F_{\text{aq}}}{V_{\text{aq}}}(C_{\text{Glu,feed}} - C_{\text{Glu}}) - k_1 C_{\text{Glu}} + k_2 C_{\text{Fru}}$$

$$\frac{dC_{\text{Fru}}}{dt} = \frac{F_{\text{aq}}}{V_{\text{aq}}}(0 - C_{\text{Fru}}) + k_1 C_{\text{Glu}} - k_2 C_{\text{Fru}} - k_3 C_{\text{Fru}}$$

$$\frac{dC_{\text{HMF,aq}}}{dt} = \frac{F_{\text{aq}}}{V_{\text{aq}}}(0 - C_{\text{HMF,aq}}) + k_3 C_{\text{Fru}} - k_4 C_{\text{HMF,aq}} - k_5 C_{\text{HMF,aq}} - J_{LL}$$

$$\frac{dC_{\text{Hum}}}{dt} = \frac{F_{\text{aq}}}{V_{\text{aq}}}(0 - C_{\text{Hum}}) + k_4 C_{\text{HMF,aq}}$$

$$\frac{dC_{\text{LA}}}{dt} = \frac{F_{\text{aq}}}{V_{\text{aq}}}(0 - C_{\text{LA}}) + k_5 C_{\text{HMF,aq}}$$

**Organic phase** (MIBK) — $J_{LL}$ rescaled from aqueous basis; $J_{\text{HMF}}$ and $J_{O_2}$ on organic basis:

$$\frac{dC_{\text{HMF,org}}}{dt} = \frac{F_{\text{org}}}{V_{\text{org}}}(0 - C_{\text{HMF,org}}) + J_{LL}\frac{V_{\text{aq}}}{V_{\text{org}}} - J_{\text{HMF}}$$

$$\frac{dC_{O_2\text{,org}}}{dt} = \frac{F_{\text{org}}}{V_{\text{org}}}(0 - C_{O_2\text{,org}}) + J_{GL} - J_{O_2}$$

**Catalyst particle** (no convective flow) — $J_{\text{HMF}}$ and $J_{O_2}$ rescaled from organic basis:

$$\frac{dC_{\text{HMF,p}}}{dt} = J_{\text{HMF}}\frac{V_{\text{org}}}{V_p} - k_6\,\eta\,C_{\text{HMF,p}}\,C_{O_2\text{,p}}$$

$$\frac{dC_{O_2\text{,p}}}{dt} = J_{O_2}\frac{V_{\text{org}}}{V_p} - k_6\,\eta\,C_{\text{HMF,p}}\,C_{O_2\text{,p}}$$

$$\frac{dC_{\text{FDCA,p}}}{dt} = k_6\,\eta\,C_{\text{HMF,p}}\,C_{O_2\text{,p}} + J_\text{FDCA,PO}\frac{V_{\text{org}}}{V_p}$$
$$\frac{dC_{\text{FDCA,org}}}{dt} = - - \frac{F_{\text{org}}}{V_{\text{org}}}C_{\text{FDCA}} - J_\text{FDCA,OP} + J_\text{FDCA,LL}\frac{V_\text{aq}}{V_\text{org}}$$
$$\frac{dC_{\text{FDCA,aq}}}{dt} = - \frac{F_{\text{aq}}}{V_{\text{aq}}}C_{\text{FDCA}} - J_\text{FDCA,LL}$$

---

#### Mass transfer fluxes $[\text{mol}\,\text{m}^{-3}_{\text{source}}\,\text{s}^{-1}]$

The driving force for each flux is the concentration difference between bulk phases, corrected
by the partition coefficient $m$ where phases are thermodynamically distinct. Setting
$m^{OP} = 1$ for both HMF and O₂ at the organic–particle interface reflects the assumption
that the catalyst pores are wetted by the same organic solvent — no thermodynamic jump occurs there.

$$J_{GL}  = k_La_{GL}\!\left(C_{O_2}^* - C_{O_2\text{,org}}\right) \qquad\qquad\qquad\qquad \text{O}_2\text{: gas} \to \text{organic}$$

$$J_{LL}  = k_La_{\text{LL,HMF}}\!\left(C_{\text{HMF,aq}} - m_{AO}\,C_{\text{HMF,org}}\right) \qquad \text{HMF: aq} \to \text{organic},\quad m_{AO} = 0.77$$

$$J_{\text{HMF}} = k_La_{\text{LS,HMF}}\!\left(C_{\text{HMF,org}} - C_{\text{HMF,p}}\right) \qquad\qquad \text{HMF: organic} \to \text{catalyst},\quad m^{OP}_{HMF} = 1$$

$$J_{O_2} = k_La_{\text{LS,O}_2}\!\left(C_{O_2\text{,org}} - C_{O_2\text{,p}}\right) \qquad\qquad\qquad\qquad \text{O}_2\text{: organic} \to \text{catalyst},\quad m^{OP}_{O_2} = 1$$

$$J_{\text{FDCA,LL}} = k_La_{\text{LL,FDCA}}\!\left(C_{\text{FDCA,aq}} - C_{\text{FDCA,org}}\right) \qquad \text{FDCA}\text{: aquatic} \to \text{organic}, \quad m^{LL}_{\text{FDCA}} = 1$$

$$J_{\text{FDCA,OP}} = k_La_{\text{LS,FDCA}}\!\left(C_{\text{FDCA,org}} - C_{\text{FDCA,p}}\right) \qquad \text{FDCA}\text{: organic} \to \text{catalyst}, \quad m^{OP}_{\text{FDCA}} = 1$$

> **Volume-ratio prefactors** — fluxes $J$ are computed per m³ of the *source* phase. When a flux
> appears in the balance of a *different* phase it is multiplied by $V_{\text{source}}/V_{\text{destination}}$
> to conserve moles. For example, $J_{LL}$ ($\text{mol}\,\text{m}^{-3}\,\text{ s}^{-1}$) is multiplied by $V_{\text{aq}}/V_{\text{org}}$
> before entering the organic-phase balance; likewise $J_{\text{HMF}}$ and $J_{O_2}$ are multiplied by
> $V_{\text{org}}/V_p$ before entering the particle balance.

In [ ]:
def odes(t, y):
    Glu, Fru, HMF_aq, HMF_org, HMF_p, O2_org, O2_p, FDCA_p, FDCA_org, FDCA_aq, Hum, LA = \
        [max(v, 0.0) for v in y]

    # Reaction rates [mol/m^3_phase/s]
    r1  = k1 * Glu                  # Glucose -> Fructose
    r2  = k2 * Fru                  # Fructose -> Glucose (reverse)
    r3  = k3 * Fru                  # Fructose -> HMF
    r4  = k4 * HMF_aq               # HMF -> Humins
    r5  = k5 * HMF_aq               # HMF -> LA + FA
    r6  = k6 * eta * HMF_p * O2_p   # HMF oxidation on catalyst (eta = effectiveness factor)

    # Mass transfer fluxes [mol/m^3_source/s]
    J_GL      = kLa_GL * (C_O2_sat - O2_org)            # O2:   gas  -> organic
    J_LL      = kLa_LL_HMF * (HMF_aq - m_AO * HMF_org)      # HMF:  aq   -> organic
    J_HMF     = kLa_LS_HMF * (HMF_org - HMF_p)          # HMF:  org  -> catalyst
    J_O2      = kLa_LS_O2 * (O2_org - O2_p)             # O2:   org  -> catalyst
    J_FDCA_LL = kLa_LL_FDCA * (FDCA_org - FDCA_p)       # FDCA: aq   -> org
    J_FDCA_LS = kLa_LS_FDCA * (FDCA_org - FDCA_p)       # FDCA: org  -> catalyst

    # Mole balances: dC/dt = (F/V)*(C_in - C) + reaction +/- mass transfer
    dGlu      = (F_aq/V_aq) * (C_Glu_feed - Glu) + (-r1 + r2)
    dFru      = (F_aq/V_aq) * (0 - Fru) + (r1 - r2 - r3)
    dHMFaq    = (F_aq/V_aq) * (0 - HMF_aq) + (r3 - r4 - r5) - J_LL
    dHMForg   = (F_org/V_org) * (0 - HMF_org) + J_LL*(V_aq/V_org) - J_HMF
    dO2org    = (F_org/V_org) * (0 - O2_org) + J_GL - J_O2
    dHMFp     = J_HMF * (V_org/V_p) - r6
    dO2p      = J_O2 * (V_org/V_p) - r6
    dFDCA_p   = r6 + J_FDCA_LS*(V_org/V_p)
    dFDCA_org = (F_org/V_org) * (0 - FDCA_org) - J_FDCA_LS + J_FDCA_LL*(V_aq/V_org)
    dFDCA_aq  = (F_aq/V_aq) * (0 - FDCA_aq) - J_FDCA_LL
    dHum      = (F_aq/V_aq) * (0 - Hum) + r4
    dLA       = (F_aq/V_aq) * (0 - LA) + r5

    return [dGlu, dFru, dHMFaq, dHMForg, dHMFp, dO2org, dO2p, dFDCA_p, dFDCA_org, dFDCA_aq, dHum, dLA]

### 8. Integration — Reaching Steady State

The BDF (Backward Differentiation Formula) solver is used. The simulation runs for 10 residence times to ensure steady state is reached.

Initial conditions: reactor filled with fresh glucose feed; O₂ at saturation throughout liquid.

In [ ]:
y0  = [C_Glu_feed, 0, 0, 0, 0, C_O2_sat, C_O2_sat, 0, 0, 0, 0, 0]
sol = solve_ivp(odes, [0, 10*tau], y0, method='BDF', rtol=1e-9, atol=1e-11, dense_output=True)

t_plot = np.linspace(0, 10*tau, 2000)
y_plot = sol.sol(t_plot) 
ss = sol.y[:, -1]

Glu_ss, Fru_ss, HMFaq_ss, HMForg_ss, HMFp_ss, O2org_ss, O2p_ss, FDCAp_ss, FDCAorg_ss, FDCAaq_ss, Hum_ss, LA_ss = ss

print(f"Integration successful: {sol.success}")
print(f"Steady-state concentrations [mol/m_phase^3]:")
print(f"  Glu      = {Glu_ss:.1f}")
print(f"  Fru      = {Fru_ss:.1f}")
print(f"  HMF_aq   = {HMFaq_ss:.3f}")
print(f"  HMF_org  = {HMForg_ss:.3f}")
print(f"  O2_org   = {O2org_ss:.4f}")
print(f"  FDCA_org = {FDCAorg_ss:.3f}")
print(f"  FDCA_aq  = {FDCAaq_ss:.3f}")
print(f"  Humins   = {Hum_ss:.3f}")
print(f"  LA=FA    = {LA_ss:.3f}")

### 9. Steady-State Performance Metrics

#### Glucose conversion
$$X_{\text{Glu}} = \frac{C_{\text{Glu,feed}} - C_{\text{Glu,SS}}}{C_{\text{Glu,feed}}}$$

#### Selectivity (molar basis)
$$S_i = \frac{R_{\text{d}}}{\Sigma_i R_{\text{u,i}}}$$

Two multiple selectivities can be calculated. The most important ones are the selectivity of $R_1, R_6$, where for $R_1$ the selectivity is only calculated over the whole reaction step, while for $R_6$ the selectivity is calculated against the whole reaction networks or the "parallel reactions" $R_4$ and $R_5$.

In [ ]:
X_Glu = 1 - Glu_ss / C_Glu_feed

r6_ss     = k6 * eta * HMFp_ss * O2p_ss
R_FDCA    = r6_ss * V_p                     # mol/s  FDCA production rate
F_Hum     = k4 * HMFaq_ss * V_aq            # mol/s  Humins formation rate ########### ADD THE REACTION RATES (mol/s) FOR ALL reactions
F_LA      = k5 * HMFaq_ss * V_aq            # mol/s  Levulinic acid rate
F_FA      = F_LA                            # mol/s  Formic acid (equimolar with LA)
F_Glu_rxd = F_aq * (C_Glu_feed - Glu_ss)

S_FDCA = R_FDCA / F_Glu_rxd
S_Hum  = F_Hum / F_Glu_rxd
S_LA   = F_LA / F_Glu_rxd

m_OP_O2  = 1 + O2p_ss/O2org_ss              # adsorption effectiveness check for O2
m_OP_HMF = 1 + O2p_ss/O2org_ss              # adsorption effectiveness check for HMF

MW_FDCA, MW_Hum, MW_LA, MW_FA = 168.11, 126.11, 116.12, 46.03   # g/mol

print("="*50)
print("STEADY-STATE RESULTS")
print("="*50)
print(f"Glucose conversion   X  = {X_Glu*100:.1f}%")
print(f"FDCA selectivity     S  = {S_FDCA*100:.1f}%")
print(f"Humins selectivity      = {S_Hum*100:.2f}%")
print(f"LA+FA selectivity       = {S_LA*100:.2f}%")
print()
print(f"FDCA   = {F_FDCA*MW_FDCA/1000*3600:.0f} kg/h")
print(f"Humins = {F_Hum*MW_Hum/1000*3600:.1f} kg/h")
print(f"LA     = {F_LA*MW_LA/1000*3600:.1f} kg/h")
print(f"FA     = {F_FA*MW_FA/1000*3600:.1f} kg/h\n")

print("Checking adsorption of individual components (expected m_OP~1)")
print(f"m_OP_O2  = {m_OP_O2:.1f}")
print(f"m_OP_HMF = {m_OP_HMF:.1f}")


### 10. Rate-Determining Step (RDS) Analysis

To identify the bottleneck in the process chain, we compute the maximum possible rate for each step assuming all upstream steps are infinitely fast (reactants accumulate to their maximum value).

The step with the smallest maximum rate is the rate-determining step.

| Step | Assumption | Max driving force |
|------|-----------|------------------|
| $k_1$ Glu→Fru | Nothing consumed | $C_{\text{Glu,feed}}$ |
| $k_3$ Fru→HMF | Iso. equilibrium | $C_{\text{Fru,max}} = \frac{k_1/k_2}{1+k_1/k_2}\,C_{\text{Glu,feed}}$ |
| LL HMF aq→org | All Glu→HMF, blocked in aq | $C_{\text{HMF,aq}} = C_{\text{Glu,feed}}$ |
| LS HMF org→cat | Partition eq. from aq | $C_{\text{HMF,org}} = m_{AO}\,C_{\text{Glu,feed}}$ |
| GL O₂ gas→org | Max Henry driving force | $C_{O_2}^*$ |
| LS O₂ org→cat | GL is fast | $C_{O_2}^*$ |
| $k_6$ reaction | Both reactants at max | $C_{\text{HMF,p}} = C_{\text{HMF,org,max}}$, $C_{O_2,p} = C_{O_2}^*$ |

In [ ]:
K_eq = k1 / k2                                      # isomerisation equilibrium constant
C_Fru_max = K_eq / (1 + K_eq) * C_Glu_feed          # max fructose (mass balance)
C_HMF_aq_max = C_Glu_feed                           # all carbon as HMF
C_HMF_org_max = m_AO * C_HMF_aq_max                 # partition equilibrium

rds = {
    'k1  Glu->Fru  (aq)':  k1 * C_Glu_feed * V_aq,
    'k3  Fru->HMF  (aq)':  k3 * C_Fru_max * V_aq,
    'LL  HMF aq->org':     kLa_LL_HMF * C_HMF_aq_max * V_aq,
    'LS  HMF org->cat':    kLa_LS_HMF * C_HMF_org_max * V_org,
    'GL  O2  gas->org':    kLa_GL * C_O2_sat * V_org,
    'LS  O2  org->cat':    kLa_LS_O2 * C_O2_sat * V_org,
    'k6*eta  reaction':    k6*eta * C_HMF_org_max * C_O2_sat * V_p,
}

print("="*55)
print("RDS -- MAXIMUM DRIVING FORCE [mol/s]")
print("="*55)
min_rate = min(rds.values())
for name, rate in rds.items():
    tag = "  <-- BOTTLENECK" if rate == min_rate else ""
    print(f"  {name:25s}  {rate:.3e}{tag}")

### 11. Results — Dynamic Profiles and Steady-State Performance

Six panels show the transient behaviour from startup to steady state. Vertical dashed lines mark each residence time $\tau$; horizontal dotted lines mark the steady-state value for each variable.

In [ ]:
t_h   = t_plot / 3600
tau_h = tau / 3600

def tau_lines(ax):
    for i in range(1, 6):
        ax.axvline(i*tau_h, color='grey', lw=0.6, ls='--', alpha=0.5)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle(
    'CSTR: Glucose -> Fructose -> HMF -> FDCA\n'
    'T = 140 C,  P = 10 bar,  tau = 1 h,  V = 100 m^3',
    fontsize=12, fontweight='bold')
plt.subplots_adjust(hspace=0.40, wspace=0.35,
                    left=0.07, right=0.97, top=0.88, bottom=0.09)

# Panel 1 -- Sugars (aqueous)
ax = axes[0, 0]
ax.plot(t_h, y_plot[0], label='Glucose')
ax.plot(t_h, y_plot[1], label='Fructose')
ax.axhline(Glu_ss, color='C0', ls=':', lw=1)
ax.axhline(Fru_ss, color='C1', ls=':', lw=1)
ax.set(xlabel='Time (h)', ylabel='Concentration (mol/m^3)', title='Sugars (aq)')
ax.legend(); ax.grid(alpha=0.3); tau_lines(ax)

# Panel 2 -- HMF across phases
ax = axes[0, 1]
ax.plot(t_h, y_plot[2],          label='HMF (aq)')
ax.plot(t_h, y_plot[3],          label='HMF (org)')
ax.plot(t_h, y_plot[4], ls='--', label='HMF (particle)')
ax.axhline(HMFaq_ss,  color='C0', ls=':', lw=1)
ax.axhline(HMForg_ss, color='C1', ls=':', lw=1)
ax.set(xlabel='Time (h)', ylabel='Concentration (mol/m^3)', title='HMF -- All Phases')
ax.legend(); ax.grid(alpha=0.3); tau_lines(ax)

# Panel 3 -- Byproducts
ax = axes[0, 2]
ax.plot(t_h, y_plot[7],          label='Humins (aq)')
ax.plot(t_h, y_plot[8], ls='--', label='LA = FA (aq)')
ax.axhline(Hum_ss, color='C0', ls=':', lw=1)
ax.axhline(LA_ss,  color='C1', ls=':', lw=1)
ax.set(xlabel='Time (h)', ylabel='Concentration (mol/m^3)', title='Byproducts (aq)')
ax.legend(); ax.grid(alpha=0.3); tau_lines(ax)

# Panel 4 -- O2
ax = axes[1, 0]
ax.plot(t_h, y_plot[5],          label='O2 (org)')
ax.plot(t_h, y_plot[6], ls='--', label='O2 (particle)')
ax.axhline(C_O2_sat, color='steelblue', ls=':', lw=1, label=f'C* = {C_O2_sat:.2f}')
ax.set(xlabel='Time (h)', ylabel='Concentration (mol/m^3)', title='O2')
ax.legend(); ax.grid(alpha=0.3); tau_lines(ax)

# Panel 5 -- Conversion and FDCA selectivity
ax  = axes[1, 1]
ax2 = ax.twinx()
X_t = (C_Glu_feed - y_plot[0]) / C_Glu_feed * 100
r6_t = k6 * eta * np.maximum(y_plot[4], 0) * np.maximum(y_plot[6], 0)
F_Glu_rxd_t = F_aq * (C_Glu_feed - y_plot[0])
with np.errstate(invalid='ignore', divide='ignore'):
    S_t = np.where(F_Glu_rxd_t > 0.01, r6_t*V_p/F_Glu_rxd_t*100, 0)
l1, = ax.plot(t_h, X_t, 'C0',          label='Glu conversion (%)')
l2, = ax2.plot(t_h, S_t,'C3', ls='--', label='FDCA selectivity (%)')
ax.axhline(X_Glu*100,   color='C0', ls=':', lw=1)
ax2.axhline(S_FDCA*100, color='C3', ls=':', lw=1)
ax.set(xlabel='Time (h)', ylabel='Conversion (%)', ylim=(0,105), title='Performance')
ax2.set_ylabel('Selectivity (%)'); ax2.set_ylim(0, 105)
ax.legend([l1, l2], [l.get_label() for l in [l1, l2]], loc='center right')
ax.grid(alpha=0.3); tau_lines(ax)

# Panel 6 -- Product rates (log scale)
ax = axes[1, 2]
r6_t_here = k6*eta*np.maximum(y_plot[4],0)*np.maximum(y_plot[6],0)
FDCA_t   = r6_t_here * V_p * 3600 * MW_FDCA/1000
Humins_t = k4 * np.maximum(y_plot[2],0) * V_aq * 3600 * MW_Hum/1000
LA_t     = k5 * np.maximum(y_plot[2],0) * V_aq * 3600 * MW_LA/1000
FA_t     = k5 * np.maximum(y_plot[2],0) * V_aq * 3600 * MW_FA/1000
ax.plot(t_h, FDCA_t,   label=f'FDCA   ({F_FDCA*MW_FDCA/1000*3600:.0f} kg/h)')
ax.plot(t_h, Humins_t, label=f'Humins ({F_Hum*MW_Hum/1000*3600:.1f} kg/h)')
ax.plot(t_h, LA_t,     ls='--', label=f'LA ({F_LA*MW_LA/1000*3600:.1f} kg/h)')
ax.plot(t_h, FA_t,     ls='-.', label=f'FA ({F_FA*MW_FA/1000*3600:.1f} kg/h)')
ax.set_yscale('log')
ax.set(xlabel='Time (h)', ylabel='Production rate (kg/h)', title='Product Rates')
ax.legend(fontsize=8.5); ax.grid(alpha=0.3); tau_lines(ax)

plt.tight_layout()
plt.show()

### 14. Parameter Study
Here we look at how small changes parameters such as $T, \varepsilon_\text{s}, \varepsilon_\text{org}$ lead to changes in the conversion and if they are impactfull in a significant way. 

1) make a list of parameters we want to test and change
2) make a range and number of points on said range to change up the parameters
3) make the odes have some parameters that are fixed through the scope insertion from outside function to inside at function initialisation.
4) other parameters are set via arguments of the function -> making them changable and can be defined to have a standard value at function creation filled in, eg. f(t,y, params, x=X)
5) plot the results and have the conclusions